In [12]:
import cv2
import numpy as np
import time
from mss import mss

# =========================
# PREPROCESSING FOR STAMPS
# =========================
stamp_paths = [
               r"C:\Osu!Conda\digits\one.jpg",
               r"C:\Osu!Conda\digits\two.jpg",
               r"C:\Osu!Conda\digits\three.jpg",
               r"C:\Osu!Conda\digits\four.jpg",
               r"C:\Osu!Conda\digits\five.jpg",
               r"C:\Osu!Conda\digits\six.jpg",
               r"C:\Osu!Conda\digits\seven.jpg",
               r"C:\Osu!Conda\digits\eight.jpg",
               r"C:\Osu!Conda\digits\nine.jpg",
               r"C:\Osu!Conda\digits\background.jpg"]

digit_stamps = ["dih"]
digit_stamps[0] = np.array((cv2.imread(r"C:\Osu!Conda\digits\zero.jpg", cv2.IMREAD_GRAYSCALE)))
for path in stamp_paths: 
    digit_stamps.append(np.array((cv2.imread(path, cv2.IMREAD_GRAYSCALE))))

start_row, end_row = 1000, 1080
start_col, end_col = 10, 60

digit_stamps[0] = digit_stamps[0][start_row:end_row, start_col+50:end_col+60]

for x in range(1,11):
    digit_stamps[x] = digit_stamps[x][start_row:end_row, start_col:end_col]

# =========================
# SCREEN CAPTURE FUNCTION
# =========================
def digit_frames():
    with mss() as sct:
        digit_screen1_area = {"top":start_row,"left":start_col,"width":50,"height":80}
        digit_screen2_area = {"top":start_row,"left":start_col+55,"width":55,"height":80}
        digit_screen3_area = {"top":start_row,"left":start_col+110,"width":55,"height":80}
        digit_screen4_area = {"top":start_row,"left":start_col+160,"width":55,"height":80}

        r1 = sct.grab(digit_screen1_area)
        r2 = sct.grab(digit_screen2_area)
        r3 = sct.grab(digit_screen3_area)
        r4 = sct.grab(digit_screen4_area)

        r_1_np = np.array(r1)
        r_2_np = np.array(r2)
        r_3_np = np.array(r3)
        r_4_np = np.array(r4)

        digits_box_1 = cv2.cvtColor(r_1_np, cv2.COLOR_BGR2GRAY)
        digits_box_2 = cv2.cvtColor(r_2_np, cv2.COLOR_BGR2GRAY)
        digits_box_3 = cv2.cvtColor(r_3_np, cv2.COLOR_BGR2GRAY)
        digits_box_4 = cv2.cvtColor(r_4_np, cv2.COLOR_BGR2GRAY)

        return (digits_box_1, digits_box_2, digits_box_3, digits_box_4)

# =========================
# CONSTANTS
# =========================
BACKGROUND_INDEX = 10
MIN_ACCEPT_SCORE = 0.75
MAX_FORWARD_JUMP = 12
MIN_MARGIN = 0.08

BREAK_LANDS_AT_OR_BELOW = 1
BREAK_CONFIRM_FRAMES = 2
BREAK_STRICT_CONF = 0.80

ROLLOVER_TOLERANCE = 3
ROLLOVER_MAX_OVERSHOOT = 15

HIST_LEN = 6
BREAK_LATCH_MS = 200
BIG_DROP_RATIO = 0.6
BIG_DROP_MIN = 8
ALLOW_BACKWARDS_DURING_LATCH = True
APPEND_ZERO_REJECT = True
APPEND_ZERO_CONF_MIN = 0.75

# =========================
# SMART CACHING STATE
# =========================
frame_counter = 0
cached_digits = [BACKGROUND_INDEX, BACKGROUND_INDEX, BACKGROUND_INDEX, BACKGROUND_INDEX]
cached_digit_confidences = [0.0, 0.0, 0.0, 0.0]
cached_digit_margins = [0.0, 0.0, 0.0, 0.0]  # NEW: cache the margin too!
digit_update_intervals = [1, 10, 100, 1000]

last_good_combo = 0
raw_hist = []
break_latch_until = 0.0

# =========================
# HELPERS
# =========================
def num_digits(n: int) -> int:
    return 1 if n == 0 else len(str(n))

def expected_rollover(prev: int) -> int:
    d = num_digits(prev)
    return 10 ** d

def is_plausible_rollover(prev: int, raw: int) -> bool:
    exp = expected_rollover(prev)
    if exp <= raw <= exp + ROLLOVER_TOLERANCE:
        return True
    if exp <= raw <= exp + ROLLOVER_MAX_OVERSHOOT:
        return True
    return False

def push_hist(raw, conf):
    global raw_hist
    raw_hist.append((raw, conf))
    if len(raw_hist) > HIST_LEN:
        raw_hist.pop(0)

def count_break_like(stable_now: int):
    tiny = 0
    bigdrop = 0
    for r, c in raw_hist:
        if c >= BREAK_STRICT_CONF:
            if r <= BREAK_LANDS_AT_OR_BELOW:
                tiny += 1
            
            if stable_now > r:
                is_significant = (
                    (stable_now - r >= BIG_DROP_MIN) or 
                    (r <= stable_now * BIG_DROP_RATIO) or
                    (stable_now < 10 and r < stable_now)
                )
                if is_significant:
                    bigdrop += 1
    return tiny, bigdrop

# =========================
# SMART CACHED COMBO READING (WITH MARGIN CHECK!)
# =========================
def read_combo_smart():
    global frame_counter, last_good_combo, break_latch_until
    global cached_digits, cached_digit_confidences, cached_digit_margins

    digits = digit_frames()
    recognized_digits = []
    digit_scores = []

    for position_idx in range(4):
        interval = digit_update_intervals[position_idx]
        
        # Should we OCR this digit this frame?
        if frame_counter % interval == 0:
            # RUN OCR for this position
            best_match_index = None
            best_match_score = -1.0
            second_best_score = -1.0

            for stamp_idx in range(11):
                similarity_map = cv2.matchTemplate(
                    digits[position_idx], 
                    digit_stamps[stamp_idx], 
                    cv2.TM_CCOEFF_NORMED
                )
                _, max_val, _, _ = cv2.minMaxLoc(similarity_map)
                
                if max_val > best_match_score:
                    second_best_score = best_match_score  # Keep track of 2nd best!
                    best_match_score = float(max_val)
                    best_match_index = stamp_idx
                elif max_val > second_best_score:
                    second_best_score = float(max_val)

            match_margin = best_match_score - second_best_score

            # Update cache for this position
            cached_digits[position_idx] = best_match_index
            cached_digit_confidences[position_idx] = best_match_score
            cached_digit_margins[position_idx] = match_margin
        
        # Use cached value
        best_match_index = cached_digits[position_idx]
        best_match_score = cached_digit_confidences[position_idx]
        match_margin = cached_digit_margins[position_idx]
        
        # CRITICAL: Apply the same stopping logic as original
        if best_match_index != BACKGROUND_INDEX and match_margin < MIN_MARGIN:
            break
        if best_match_index == BACKGROUND_INDEX:
            # Invalidate higher positions
            for higher_pos in range(position_idx, 4):
                cached_digits[higher_pos] = BACKGROUND_INDEX
                cached_digit_confidences[higher_pos] = 0.0
                cached_digit_margins[higher_pos] = 0.0
            break
        
        recognized_digits.append(best_match_index)
        digit_scores.append(best_match_score)

    # 2. Build Raw Value
    if not recognized_digits:
        raw_combo = 0
        confidence = 0.0
    else:
        raw_combo = int("".join(str(d) for d in recognized_digits))
        confidence = float(min(digit_scores))

    if len(recognized_digits) >= 2 and all(d == 0 for d in recognized_digits):
        confidence = 0.0

    push_hist(raw_combo, confidence)

    # 3. Filter into Stable Combo (SAME AS ORIGINAL)
    stable_combo = last_good_combo
    accepted = False
    now = time.time()
    in_latch = now < break_latch_until

    if confidence >= MIN_ACCEPT_SCORE:
        if (APPEND_ZERO_REJECT and confidence >= APPEND_ZERO_CONF_MIN and 
            last_good_combo >= 1 and raw_combo == last_good_combo * 10):
            pass
        
        elif raw_combo == last_good_combo:
            accepted = True
        
        elif raw_combo > last_good_combo:
            jump = raw_combo - last_good_combo
            if jump <= MAX_FORWARD_JUMP:
                stable_combo = raw_combo
                accepted = True
            elif is_plausible_rollover(last_good_combo, raw_combo):
                stable_combo = raw_combo
                accepted = True

        else:
            if ALLOW_BACKWARDS_DURING_LATCH and in_latch:
                stable_combo = raw_combo
                accepted = True
            else:
                tiny_cnt, bigdrop_cnt = count_break_like(last_good_combo)
                break_confirmed = (tiny_cnt >= BREAK_CONFIRM_FRAMES) or (bigdrop_cnt >= BREAK_CONFIRM_FRAMES)

                if break_confirmed:
                    stable_combo = raw_combo
                    accepted = True
                    break_latch_until = now + (BREAK_LATCH_MS / 1000.0)
                elif (raw_combo <= BREAK_LANDS_AT_OR_BELOW and 
                      last_good_combo > BREAK_LANDS_AT_OR_BELOW and 
                      confidence >= 0.95):
                    stable_combo = raw_combo
                    accepted = True
                    break_latch_until = now + (BREAK_LATCH_MS / 1000.0)

    if accepted:
        last_good_combo = stable_combo

    return last_good_combo, raw_combo, confidence

# =========================
# TEST LOOP
# =========================
if __name__ == "__main__":
    print("Starting combo reader test... Switch to osu!")
    time.sleep(5)
    
    TARGET_FPS = 24
    FRAME_TIME = 1.0 / TARGET_FPS
    
    try:
        while True:
            loop_start = time.perf_counter()
            
            stable, raw, conf = read_combo_smart()
            
            print(f"Frame {frame_counter:4d} | RAW={raw:4d} CONF={conf:.3f} | STABLE={stable:4d} | Cache: {cached_digits}")
            
            frame_counter += 1
            
            loop_end = time.perf_counter()
            elapsed = loop_end - loop_start
            sleep_time = FRAME_TIME - elapsed
            
            if sleep_time > 0:
                time.sleep(sleep_time)
            else:
                print(f"  ⚠️  Loop lag! Over by {abs(sleep_time):.4f}s")
            
    except KeyboardInterrupt:
        print("\nTest stopped by user.")
        print(f"Final combo: {last_good_combo}")

Starting combo reader test... Switch to osu!
Frame    0 | RAW=   7 CONF=0.879 | STABLE=   7 | Cache: [7, 10, 10, 10]
Frame    1 | RAW=   8 CONF=0.986 | STABLE=   8 | Cache: [8, 10, 10, 10]
Frame    2 | RAW=   8 CONF=0.845 | STABLE=   8 | Cache: [8, 10, 10, 10]
Frame    3 | RAW=   9 CONF=0.990 | STABLE=   9 | Cache: [9, 10, 10, 10]
Frame    4 | RAW=   1 CONF=1.000 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame    5 | RAW=   1 CONF=1.000 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame    6 | RAW=   1 CONF=0.766 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame    7 | RAW=   1 CONF=1.000 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame    8 | RAW=   1 CONF=0.766 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame    9 | RAW=   1 CONF=1.000 | STABLE=   1 | Cache: [1, 10, 10, 10]
Frame   10 | RAW=   1 CONF=0.803 | STABLE=   1 | Cache: [1, 3, 10, 10]
Frame   11 | RAW=   1 CONF=0.830 | STABLE=   1 | Cache: [1, 3, 10, 10]
Frame   12 | RAW=   1 CONF=0.908 | STABLE=   1 | Cache: [1, 3, 10, 10]
Frame   13 | RAW=   1 